In [5]:
import os
import random
import string
import cv2
import numpy as np
from tensorflow.keras.models import load_model

from datetime import datetime, timedelta
from functools import wraps


from flask import Flask, request, jsonify
from werkzeug.security import generate_password_hash, check_password_hash
from pymongo import MongoClient, errors
import jwt
from bson import ObjectId
from dotenv import load_dotenv
from flask_mail import Mail, Message # Import Mail and Message
from DLModelfinctions import preprocess_for_inference,normalize_image,apply_clahe,resize_image

# --- Load Environment Variables ---
load_dotenv()
MONGO_URI = os.environ.get("MONGO_URI")
SECRET_KEY = os.environ.get("SECRET_KEY")

# --- Initialize Flask and Flask-Mail ---
app = Flask(__name__)
app.config["SECRET_KEY"] = SECRET_KEY

# Mail configuration (reads from .env)
app.config['MAIL_SERVER'] = os.environ.get('MAIL_SERVER')
app.config['MAIL_PORT'] = int(os.environ.get('MAIL_PORT', 587))
app.config['MAIL_USE_TLS'] = os.environ.get('MAIL_USE_TLS', 'True').lower() in ['true', 'on', '1']
app.config['MAIL_USERNAME'] = os.environ.get('MAIL_USERNAME')
app.config['MAIL_PASSWORD'] = os.environ.get('MAIL_PASSWORD')
app.config['MAIL_DEFAULT_SENDER'] = os.environ.get('MAIL_USERNAME')

mail = Mail(app) # Initialize Mail with the app

# --- Initialize MongoDB ---
client = MongoClient(MONGO_URI)
db = client.get_default_database()
users_col = db.users
# NEW: Collection for temporary user data
unverified_users_col = db.unverified_users



# ---------------- Load trained model ----------------
model = load_model("DenseNet2d256.h5")  # update path if needed

# ---------------- Classes ----------------
class_names = ["Early_Blight", "Healthy", "Late_Blight"]

# Folder to temporarily save uploaded images
UPLOAD_FOLDER = "./uploads"
os.makedirs(UPLOAD_FOLDER, exist_ok=True)



# --- Create Indexes for Collections ---
try:
    # This index is crucial for the main users collection
    users_col.create_index("email", unique=True)
    # This index will automatically delete temporary users after 1 hour (3600 seconds)
    unverified_users_col.create_index("created_at", expireAfterSeconds=3600)
except errors.OperationFailure as e:
    print(f"Index creation failed (this might be okay if they already exist): {e}")


# --- Helper Functions (token generation, decorators, etc.) ---

# (generate_token and token_required functions remain the same as your original code)
def generate_token(user_id: str, expires_hours: int = 24):
    payload = {
        "user_id": user_id,
        "exp": datetime.utcnow() + timedelta(hours=expires_hours),
        "iat": datetime.utcnow(),
    }
    token = jwt.encode(payload, app.config["SECRET_KEY"], algorithm="HS256")
    if isinstance(token, bytes):
        token = token.decode("utf-8")
    return token

def token_required(f):
    @wraps(f)
    def decorated(*args, **kwargs):
        auth_header = request.headers.get("Authorization", None)
        if not auth_header:
            return jsonify({"error": "Authorization header missing"}), 401

        parts = auth_header.split()
        if len(parts) != 2 or parts[0].lower() != "bearer":
            return jsonify({"error": "Authorization header must be 'Bearer <token>'"}), 401

        token = parts[1]
        try:
            payload = jwt.decode(token, app.config["SECRET_KEY"], algorithms=["HS256"])
            user_id = payload.get("user_id")
            if not user_id:
                raise jwt.InvalidTokenError()
        except jwt.ExpiredSignatureError:
            return jsonify({"error": "Token expired"}), 401
        except jwt.InvalidTokenError:
            return jsonify({"error": "Invalid token"}), 401
        
        user = users_col.find_one({"_id": ObjectId(user_id)}, {"password": 0})
        if not user:
            return jsonify({"error": "User not found"}), 401

        user["user_id"] = str(user["_id"])
        del user["_id"]
        if "created_at" in user and hasattr(user["created_at"], "isoformat"):
            user["created_at"] = user["created_at"].isoformat()

        request.user = user
        return f(*args, **kwargs)
    return decorated


# --- NEW: Helper function to generate a random code ---
def generate_verification_code(length=6):
    """Generates a random 6-digit code."""
    return "".join(random.choices(string.digits, k=length))


# --- API Endpoints ---

@app.route("/")
def home():
    return jsonify({"message": "Flask Auth API running with verification"})


# --- MODIFIED: /auth/register Endpoint ---
@app.route("/auth/register", methods=["POST"])
def register():
    data = request.get_json() or {}
    username = data.get('username')
    email = data.get('email')
    password = data.get('password')

    if not username or not email or not password:
        return jsonify({"error": "username, email and password are required"}), 400
    if len(password) < 6:
        return jsonify({"error": "password must be at least 6 characters"}), 400

    # 1. Check if user is already in the main users collection
    if users_col.find_one({"email": email}):
        return jsonify({"error": "email already registered"}), 409 # 409 Conflict

    # 2. Generate code and hash password
    verification_code = generate_verification_code()
    hashed_password = generate_password_hash(password)

    # 3. Store in the temporary 'unverified_users' collection
    # Use 'upsert' to handle cases where a user re-requests a code
    unverified_users_col.update_one(
        {"email": email},
        {
            "$set": {
                "username": username,
                "password": hashed_password,
                "code": verification_code,
                "created_at": datetime.utcnow()
            }
        },
        upsert=True
    )

    # 4. Send the verification email
    try:
        msg = Message(
            subject="Your DeepBlight Verification Code",
            recipients=[email],
            body=f"Welcome to DeepBlight! Your verification code is: {verification_code}\n\nThis code will expire in 1 hour."
        )
        mail.send(msg)
    except Exception as e:
        print(f"Failed to send email: {e}")
        return jsonify({"error": "Could not send verification email. Please try again later."}), 500

    # 5. Return success message to the Android app
    return jsonify({"message": "Verification code sent to email."}), 201


# --- NEW: /auth/verify Endpoint ---
@app.route("/auth/verify", methods=["POST"])
def verify():
    data = request.get_json() or {}
    email = data.get('email')
    code = data.get('code')

    if not email or not code:
        return jsonify({"error": "email and code are required"}), 400

    # 1. Find the user in the temporary collection
    temp_user = unverified_users_col.find_one({"email": email, "code": code})

    if not temp_user:
        return jsonify({"error": "Invalid email or verification code."}), 404 # 404 Not Found

    # 2. Create the permanent user document
    user_doc = {
        "username": temp_user["username"],
        "email": temp_user["email"],
        "password": temp_user["password"],
        "created_at": datetime.utcnow(),
    }

    # 3. Insert into the main 'users' collection
    try:
        result = users_col.insert_one(user_doc)
    except errors.DuplicateKeyError:
        # This can happen in a race condition if user verifies on two devices at once
        return jsonify({"error": "This email has just been registered."}), 409

    # 4. Clean up: Delete the record from the temporary collection
    unverified_users_col.delete_one({"email": email})

    # 5. Return success. The user can now log in.
    return jsonify({"message": "User verified successfully. Please log in."}), 200


# --- UNCHANGED: /auth/login and /profile Endpoints ---
@app.route("/auth/login", methods=["POST"])
def login():
    data = request.get_json() or {}
    email = (data.get("email") or "").lower().strip()
    password = data.get("password") or ""

    if not email or not password:
        return jsonify({"error": "email and password are required"}), 400

    user = users_col.find_one({"email": email})
    if not user:
        return jsonify({"error": "invalid email or password"}), 401

    if not check_password_hash(user["password"], password):
        return jsonify({"error": "invalid email or password"}), 401

    user_id = str(user["_id"])
    token = generate_token(user_id)
    user_data = {"user_id": user_id, "username": user.get("username"), "email": user.get("email")}
    return jsonify({"message": "login successful", "token": token, "user": user_data})


@app.route("/profile", methods=["GET"])
@token_required
def profile():
    user = request.user
    return jsonify({"message": "profile", "user": user})


# ----------- Endpoint for image upload -----------
@app.route("/upload", methods=["POST"])
def upload_image():
    if "image" not in request.files:
        return jsonify({"error": "No image file provided"}), 400

    file = request.files["image"]
    if file.filename == "":
        return jsonify({"error": "Empty filename"}), 400

    # Save the uploaded image temporarily
    img_path = os.path.join(UPLOAD_FOLDER, file.filename)
    file.save(img_path)

    # Process and run inference
    processed_img = preprocess_for_inference(img_path)
    pred_probs = model.predict(processed_img)
    pred_class = np.argmax(pred_probs, axis=1)[0]

    # Print results on server console
    print(f"Predicted class: {class_names[pred_class]}")
    print(f"Class probabilities: {pred_probs[0]}")

    # Prepare JSON response for Android app
    response = {
        "predicted_class": class_names[pred_class],
        "probabilities": pred_probs[0].tolist()
    }

    return jsonify(response), 200


if __name__ == "__main__":
    app.run(host='0.0.0.0', port=5000, debug=True)



ModuleNotFoundError: No module named 'cv2'

In [ ]:
pip install opencv-python
